__Dataset__: [Link](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)

In [1]:
# Download the dataset

# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")

# print("Path to dataset files:", path)

In [77]:
import numpy as np
import pandas as pd
import re
import string
from utils.constants import chat_words
import os

import nltk
NLTK_DIR = os.path.abspath("./env/nltk_data")
# Tokenizer
nltk.download('punkt', download_dir=NLTK_DIR, raise_on_error=True)
nltk.download('punkt_tab', download_dir=NLTK_DIR, raise_on_error=True)
# Correct spelling
nltk.download('words', download_dir=NLTK_DIR, raise_on_error=True)
# stopword removal
nltk.download('stopwords', download_dir=NLTK_DIR, raise_on_error=True)
# download wordnet
nltk.download('wordnet', download_dir=NLTK_DIR, raise_on_error=True)

nltk.data.path.insert(0, NLTK_DIR)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.metrics.distance import jaccard_distance
from nltk.util import ngrams
from nltk.corpus import words, stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer



[nltk_data] Downloading package punkt to /home/sudipto/projects/test-
[nltk_data]     projects/sms_spam/env/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/sudipto/projects/test-
[nltk_data]     projects/sms_spam/env/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/sudipto/projects/test-
[nltk_data]     projects/sms_spam/env/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package words to /home/sudipto/projects/test-
[nltk_data]     projects/sms_spam/env/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/sudipto/projects/test-
[nltk_data]     projects/sms_spam/env/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/sud

In [3]:
DATAPATH = 'data/spam.csv'

df = pd.read_csv(
    DATAPATH,
    header=0,
    usecols=[0, 1],
    names=["label", "message"],
    encoding="utf-8",
    encoding_errors="ignore"
)
df = df.dropna(subset=["label", "message"]).copy()
df["label"] = df["label"].astype(str).str.strip().str.lower()
df["message"] = df["message"].astype(str)

print(f"Messages: {len(df):,}")
print("Label counts:", df["label"].value_counts().to_dict())
df.sample(5)


Messages: 5,572
Label counts: {'ham': 4825, 'spam': 747}


,label,message
3409,ham,Joy's father is John. Then John is the ____ of...
847,ham,I am in office:)whats the matter..msg me now.i...
3952,spam,Refused a loan? Secured or Unsecured? Can't ge...
4550,ham,Great. Have a safe trip. Dont panic surrender ...
3749,ham,"Buzz! Hey, my Love ! I think of you and hope y..."


In [88]:

def remove_htmltags(text):
    pattern = re.compile(r"<.*?>")
    return pattern.sub(r"", text)

def remove_urls(text):
    pattern = re.compile(r"https?:\/\/\S+|www\.\S+")
    return pattern.sub(r"", text)

def remove_punctuations(text):
    excluded_punctuations = string.punctuation
    return text.translate(str.maketrans(" ", " ", excluded_punctuations))

def eloborate_chat_conversion(texts):
    new_text = []
    token_words = texts.split()
    
    for word in token_words:
        key = word.upper()
        if key in chat_words:
            new_text.append(chat_words[key])
        else:
            new_text.append(word)
    return " ".join(new_text)


def corrected_spelling(texts):
    results = []
    correct_words = [w.lower() for w in words.words()]
    correct_set = set(correct_words)
    text_tokens = texts.split()
    
    for word in text_tokens:
        lower_word = word.lower()
        # keep the correctly spelled words
        if lower_word in correct_set or not lower_word.isalpha() or len(lower_word) < 2:
            results.append(lower_word)
            continue
        
        incorrect_bigrams = set(ngrams(lower_word, 2))
        best_score, best_match = 1.0, word
        
        for w in correct_words:
            if w[0] == lower_word[0]:
                correct_bigrams = set(ngrams(w, 2))
                score = jaccard_distance(incorrect_bigrams, correct_bigrams)
                if score < best_score:
                    best_score, best_match = score, w
        results.append(best_match)
    return " ".join(results)

def remove_stopwords(texts):
    cleaned = []
    text_tokens = texts.split()
    stop_words = set(stopwords.words('english'))
    for word in text_tokens:
        # word is already lowered
        if word and word not in stop_words:
            cleaned.append(word)
    return " ".join(cleaned)


def remove_emoji(texts):
    text_tokens = texts.split()
    text = " ".join(text_tokens)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002700-\U000027BF"  # dingbats
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )
    texts = emoji_pattern.sub(r'', text)
    return texts


def stemming_words(text_tokens):
    ps = PorterStemmer()
    stemmed_words = []
    return [stemmed_words.append(ps.stem(word)) for word in text_tokens]

def lemmatize_words(text_tokens):
    lemmatized_words = []
    wordnet_lemmatizer = WordNetLemmatizer()
    for word in text_tokens:
        lemmatized_words.append(wordnet_lemmatizer.lemmatize(word, pos='v'))
    return lemmatized_words


def text_preprocessing(text):
    # lowercasing
    text = text.lower()
    
    # remove html tags
    text = remove_htmltags(text)
    
    # remove urls from text
    text = remove_urls(text)
    
    text = remove_punctuations(text)
    
    text = eloborate_chat_conversion(text)
    
    text = corrected_spelling(text)
    
    text = remove_stopwords(text)
    
    text = remove_emoji(text)
    
    tokenized_text = word_tokenize(text)
    
    # tokenized_text = stemming_words(tokenized_text)
    
    tokenized_text = lemmatize_words(tokenized_text)
    
    
    return tokenized_text
    



In [89]:
text_preprocessing("""helo amzing ADIH <p>hello world</p> <br/> https://helo_texts.com
https://www.helo_texts.com
http://localhost:5000"""+df['message'][0])


['hello',
 'amaze',
 'another',
 'day',
 'hell',
 'hello',
 'world',
 'juror',
 'point',
 'crazy',
 'available',
 'bugi',
 'n',
 'great',
 'world',
 'la',
 'e',
 'buffet',
 'cine',
 'get',
 'amores',
 'wat']